# M5 Forecasting — California Subset Analysis

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
import os

CACHE_FILE = "df_CA_cache.parquet"

if os.path.exists(CACHE_FILE):
    df_CA = pd.read_parquet(CACHE_FILE)
    _from_cache = True
    print(f"Loaded from cache: {df_CA.shape} — skip to Section 4.")
    print(f"Memory: {df_CA.memory_usage(deep=True).sum() / 1e6:.1f} MB")
else:
    _from_cache = False
    print("No cache found — run sections 1–3 to build df_CA.")

Loaded from cache: (23330948, 23) — skip to Section 4.
Memory: 2010.4 MB


## 1. Load Sales Data & Extract CA Subset

In [3]:
if not _from_cache:
    df_sales = pd.read_csv("sales_train_validation.csv")

    df_sales_CA = df_sales[df_sales["state_id"] == "CA"].copy()
    del df_sales

    id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
    sales_cols = [col for col in df_sales_CA.columns if col.startswith("d_")]

    df_CA_long = df_sales_CA.melt(
        id_vars=id_cols,
        value_vars=sales_cols,
        var_name="d",
        value_name="sales"
    )
    del df_sales_CA

    df_CA_long["d_num"] = df_CA_long["d"].str.extract(r"d_(\d+)").astype("uint16")

    # "id" stays as object — used for EDA, will be dropped before training
    df_CA_long = df_CA_long.astype({
        "item_id":  "category",
        "dept_id":  "category",
        "cat_id":   "category",
        "store_id": "category",
        "state_id": "category",
        "d":        "category",
        "sales":    "uint16",
    })

    print("CA long shape:", df_CA_long.shape)
    print(f"Memory: {df_CA_long.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 2. Load Calendar & Merge with CA Data

In [ ]:
if not _from_cache:
    df_calendar = pd.read_csv("calendar.csv")

    for col in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
        df_calendar[col] = df_calendar[col].fillna("No_Event")

    df_calendar = df_calendar.astype({
        "wm_yr_wk":     "uint16",
        "wday":         "uint8",
        "month":        "uint8",
        "year":         "uint16",
        "weekday":      "category",
        "event_name_1": "category",
        "event_type_1": "category",
        "event_name_2": "category",
        "event_type_2": "category",
        "snap_CA": "bool",
        "snap_TX": "bool",
        "snap_WI": "bool",
    })
    df_calendar["date"] = pd.to_datetime(df_calendar["date"]).astype("datetime64[ms]")

    df_CA_merged = df_CA_long.merge(df_calendar, on="d", how="left")
    del df_calendar

    print("df_CA_merged shape:", df_CA_merged.shape)
    print(f"Memory: {df_CA_merged.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 3. Load Sell Prices & Merge with CA Data

In [ ]:
if not _from_cache:
    df_sell_prices = pd.read_csv("sell_prices.csv")

    df_sell_prices_CA = df_sell_prices[df_sell_prices["store_id"].str.startswith("CA")].copy()
    del df_sell_prices

    df_sell_prices_CA = df_sell_prices_CA.astype({
        "store_id":   "category",
        "item_id":    "category",
        "wm_yr_wk":   "uint16",
        "sell_price": "float32",
    })

    df_CA = df_CA_merged.merge(
        df_sell_prices_CA,
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left"
    )
    del df_CA_merged, df_sell_prices_CA

    df_CA.to_parquet(CACHE_FILE, index=False)
    print(f"Cache saved → {CACHE_FILE}")

print("df_CA shape:", df_CA.shape)
print("Stores:", sorted(df_CA["store_id"].unique()))
print(f"Memory: {df_CA.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("\nNull counts:")
print(df_CA.isnull().sum()[df_CA.isnull().sum() > 0])
df_CA.head()

## 4. Explore df_CA

In [6]:
print(df_CA.dtypes)
print("\nDate range:", df_CA["date"].min(), "→", df_CA["date"].max())
print("d_num range:", df_CA["d_num"].min(), "→", df_CA["d_num"].max())

id                         str
item_id               category
dept_id               category
cat_id                category
store_id              category
state_id              category
d                          str
sales                   uint16
d_num                   uint16
date            datetime64[ms]
wm_yr_wk                uint16
weekday               category
wday                     uint8
month                    uint8
year                    uint16
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap_CA                   bool
snap_TX                   bool
snap_WI                   bool
sell_price             float32
dtype: object

Date range: 2011-01-29 00:00:00 → 2016-04-24 00:00:00
d_num range: 1 → 1913


## Plot 1: Total Daily Sales — California

In [ ]:
daily_sales = df_CA.groupby("date")["sales"].sum()

plt.figure(figsize=(14, 4))
plt.plot(daily_sales.index, daily_sales.values, linewidth=0.8, color="steelblue")
plt.title("Total Daily Sales — California")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("ca_daily_sales.png", dpi=150)
plt.show()

## Plot 2: Daily Sales by CA Store

In [ ]:
store_daily = df_CA.groupby(["date", "store_id"])["sales"].sum().reset_index()
stores = sorted(df_CA["store_id"].unique())

fig, axes = plt.subplots(len(stores), 1, figsize=(14, 4 * len(stores)), sharex=True)

for ax, store in zip(axes, stores):
    grp = store_daily[store_daily["store_id"] == store]
    ax.plot(grp["date"], grp["sales"], linewidth=0.8)
    ax.set_title(store)
    ax.set_ylabel("Units Sold")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Date")
fig.suptitle("Daily Sales by CA Store", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("ca_sales_by_store.png", dpi=150)
plt.show()

## Plot 3: Daily Sales by Category — California

In [ ]:
cat_daily = df_CA.groupby(["date", "cat_id"])["sales"].sum().reset_index()

plt.figure(figsize=(14, 5))
for cat, grp in cat_daily.groupby("cat_id"):
    plt.plot(grp["date"], grp["sales"], label=cat, linewidth=0.8)

plt.title("Daily Sales by Category — California")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("ca_sales_by_category.png", dpi=150)
plt.show()

## Plot 4: Total Sales by Department — California

In [ ]:
dept_totals = df_CA.groupby("dept_id")["sales"].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
dept_totals.plot(kind="bar", color="steelblue", edgecolor="black")
plt.title("Total Sales by Department — California")
plt.xlabel("Department")
plt.ylabel("Total Units Sold")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("ca_sales_by_dept.png", dpi=150)
plt.show()

## Plot 5: SNAP Day Effect on CA Sales

In [ ]:
snap_effect = df_CA.groupby("snap_CA")["sales"].mean()

plt.figure(figsize=(5, 4))
snap_effect.plot(kind="bar", color=["#4c72b0", "#dd8452"], edgecolor="black")
plt.title("Avg Sales: SNAP Day vs Non-SNAP (CA)")
plt.ylabel("Avg Units Sold")
plt.xticks([0, 1], ["No SNAP", "SNAP Day"], rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("ca_snap_effect.png", dpi=150)
plt.show()

## Plot 6: Average Sell Price Over Time — California

In [ ]:
price_trend = df_CA.groupby("date")["sell_price"].mean()

plt.figure(figsize=(14, 4))
plt.plot(price_trend.index, price_trend.values, linewidth=0.8, color="darkorange")
plt.title("Average Sell Price Over Time — California")
plt.xlabel("Date")
plt.ylabel("Avg Sell Price ($)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("ca_price_trend.png", dpi=150)
plt.show()

## 7. Stationarity Check (ADF + KPSS)

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

# Test on one representative series per category to keep it fast
# pandas 3.0: groupby().apply() drops group key columns; use sort+dedup instead
sample_series = (
    df_CA.groupby(["cat_id", "item_id", "store_id"], observed=True)["sales"]
    .sum()
    .reset_index()
    .sort_values("sales", ascending=False)
    .drop_duplicates(subset="cat_id")
    .reset_index(drop=True)
)

results = []
for _, row in sample_series.iterrows():
    series = (
        df_CA[
            (df_CA["item_id"] == row["item_id"]) &
            (df_CA["store_id"] == row["store_id"])
        ]
        .sort_values("date")["sales"]
        .values.astype(float)
    )

    adf_stat, adf_p, _, _, _, _ = adfuller(series, autolag="AIC")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kpss_stat, kpss_p, _, _ = kpss(series, regression="c", nlags="auto")

    # ADF: H0 = unit root (non-stationary) → p < 0.05 means stationary
    # KPSS: H0 = stationary               → p < 0.05 means non-stationary
    adf_result  = "Stationary" if adf_p < 0.05 else "Non-stationary"
    kpss_result = "Non-stationary" if kpss_p < 0.05 else "Stationary"
    verdict = adf_result if adf_result == kpss_result else "Conflicting"

    results.append({
        "cat_id": row["cat_id"], "item_id": row["item_id"], "store_id": row["store_id"],
        "ADF p-value": round(adf_p, 4), "ADF result": adf_result,
        "KPSS p-value": round(kpss_p, 4), "KPSS result": kpss_result,
        "Verdict": verdict,
    })

print(pd.DataFrame(results).to_string(index=False))

In [ ]:
# Rolling mean + std — flat lines = stationary
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)

for ax, row in zip(axes, results):
    series = (
        df_CA[
            (df_CA["item_id"] == row["item_id"]) &
            (df_CA["store_id"] == row["store_id"])
        ]
        .sort_values("date")[["date", "sales"]]
        .set_index("date")["sales"]
        .astype(float)
    )
    roll = series.rolling(28)
    ax.plot(series.index, series.values, alpha=0.4, label="Sales")
    ax.plot(roll.mean(), label="28d mean", linewidth=1.5)
    ax.plot(roll.std(),  label="28d std",  linewidth=1.5, linestyle="--")
    ax.set_title(f"{row['item_id']} | {row['store_id']} — {row['Verdict']}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ca_stationarity_rolling.png", dpi=150)
plt.show()

## 7b. Stationarity Check — Stratified by Zero-Rate

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

# Step 1: compute zero-rate per item x store
zero_rate = (
    df_CA.groupby(["item_id", "store_id"], observed=True)
    .apply(lambda x: (x["sales"] == 0).mean(), include_groups=False)
    .reset_index(name="zero_rate")
)

# Attach cat_id
zero_rate = zero_rate.merge(
    df_CA[["item_id", "store_id", "cat_id"]].drop_duplicates(),
    on=["item_id", "store_id"]
)

# Step 2: stratify into regular vs intermittent
zero_rate["bucket"] = pd.cut(
    zero_rate["zero_rate"],
    bins=[0, 0.5, 1.0],
    labels=["regular", "intermittent"],
    include_lowest=True
)

# Step 3: sample 1 item per cat_id x bucket
# pandas 3.0: apply drops group keys — merge them back from the index
sample_strat = (
    zero_rate.groupby(["cat_id", "bucket"], observed=True)
    .apply(lambda x: x.sample(n=1, random_state=42))
    .reset_index(level=[0, 1])
    .reset_index(drop=True)
)

print("Sampled series (cat_id x bucket):")
print(sample_strat[["cat_id", "bucket", "item_id", "store_id", "zero_rate"]].to_string(index=False))
print()

# Step 4: run ADF + KPSS on each sampled series
def test_stationarity(series):
    adf_stat, adf_p, *_ = adfuller(series, autolag="AIC")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    adf_result  = "Stationary"     if adf_p  < 0.05 else "Non-stationary"
    kpss_result = "Non-stationary" if kpss_p < 0.05 else "Stationary"
    verdict     = adf_result if adf_result == kpss_result else "Conflicting"
    return pd.Series({
        "ADF p": round(adf_p, 4), "ADF": adf_result,
        "KPSS p": round(kpss_p, 4), "KPSS": kpss_result,
        "Verdict": verdict
    })

results_strat = []
for _, row in sample_strat.iterrows():
    series = (
        df_CA[
            (df_CA["item_id"]  == row["item_id"]) &
            (df_CA["store_id"] == row["store_id"])
        ]
        .sort_values("date")["sales"]
        .values.astype(float)
    )
    stats = test_stationarity(series)
    results_strat.append({
        "cat_id":   row["cat_id"],
        "bucket":   row["bucket"],
        "item_id":  row["item_id"],
        "store_id": row["store_id"],
        "zero_rate": round(row["zero_rate"], 3),
        **stats.to_dict()
    })

df_results = pd.DataFrame(results_strat)
print(df_results.to_string(index=False))


## 7c. Stationarity — Population Estimate with Confidence Intervals

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from scipy import stats
import warnings

SAMPLE_PER_STRATUM = 30
CONFIDENCE = 0.95

# Step 1: zero-rate + cat_id per item x store
zero_rate_full = (
    df_CA.groupby(["item_id", "store_id"], observed=True)
    .apply(lambda x: (x["sales"] == 0).mean(), include_groups=False)
    .reset_index(name="zero_rate")
    .merge(
        df_CA[["item_id", "store_id", "cat_id"]].drop_duplicates(),
        on=["item_id", "store_id"]
    )
)
zero_rate_full["bucket"] = pd.cut(
    zero_rate_full["zero_rate"],
    bins=[0, 0.5, 1.0],
    labels=["regular", "intermittent"],
    include_lowest=True
)

print("Stratum sizes (population):")
print(zero_rate_full.groupby(["cat_id", "bucket"], observed=True).size().to_string())
print()

# Step 2: sample min(30, stratum_size) per stratum
sample = (
    zero_rate_full.groupby(["cat_id", "bucket"], observed=True)
    .apply(lambda x: x.sample(n=min(SAMPLE_PER_STRATUM, len(x)), random_state=42))
    .reset_index(level=[0, 1])
    .reset_index(drop=True)
)

# Step 3: run ADF + KPSS, classify verdict
def run_tests(series):
    try:
        _, adf_p, *_ = adfuller(series, autolag="AIC")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            _, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
        adf_s  = adf_p  < 0.05
        kpss_s = kpss_p >= 0.05
        if adf_s and kpss_s:
            return "Stationary"
        elif not adf_s and not kpss_s:
            return "Non-stationary"
        else:
            return "Conflicting"
    except Exception:
        return None

print(f"Running tests on {len(sample)} series...")
sample["verdict"] = [
    run_tests(
        df_CA[(df_CA["item_id"] == r["item_id"]) & (df_CA["store_id"] == r["store_id"])]
        .sort_values("date")["sales"].values.astype(float)
    )
    for _, r in sample.iterrows()
]
sample = sample[sample["verdict"].notna()].copy()

# Step 4: Wilson CI on proportion non-stationary (NS + Conflicting)
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = (z * (p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5) / denom
    return round(p, 3), round(center - margin, 3), round(center + margin, 3)

print(f"{'Stratum':<28} {'n':>4}  {'Stationary':>10}  {'Conflicting':>11}  {'Non-stat':>8}  {'Non-stat % (95% CI)':>22}  Conclusion")
print("-" * 105)

ci_rows = []
for (cat, bucket), grp in sample.groupby(["cat_id", "bucket"], observed=True):
    n         = len(grp)
    n_s       = (grp["verdict"] == "Stationary").sum()
    n_c       = (grp["verdict"] == "Conflicting").sum()
    n_ns      = (grp["verdict"] == "Non-stationary").sum()
    p, lo, hi = wilson_ci(n_ns + n_c, n)
    conclusion = ("Likely Non-stationary" if lo > 0.5
                  else "Likely Stationary"  if hi < 0.5
                  else "Uncertain")
    label = f"{cat}/{bucket}"
    print(f"{label:<28} {n:>4}  {n_s:>10}  {n_c:>11}  {n_ns:>8}  {p*100:>6.1f}% [{lo*100:.1f}%, {hi*100:.1f}%]  {conclusion}")
    ci_rows.append({"stratum": label, "p": p, "lo": lo, "hi": hi,
                    "Stationary": n_s, "Conflicting": n_c, "Non-stationary": n_ns, "n": n})

# Step 5: visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: stacked bar — verdict breakdown per stratum
labels  = [r["stratum"] for r in ci_rows]
s_vals  = [r["Stationary"]     / r["n"] * 100 for r in ci_rows]
c_vals  = [r["Conflicting"]    / r["n"] * 100 for r in ci_rows]
ns_vals = [r["Non-stationary"] / r["n"] * 100 for r in ci_rows]
x = range(len(labels))

axes[0].bar(x, s_vals, label="Stationary", color="#2ecc71", edgecolor="black")
axes[0].bar(x, c_vals, bottom=s_vals, label="Conflicting", color="#f39c12", edgecolor="black")
axes[0].bar(x, ns_vals, bottom=[s+c for s,c in zip(s_vals, c_vals)],
            label="Non-stationary", color="#e74c3c", edgecolor="black")
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(labels, rotation=30, ha="right")
axes[0].set_ylabel("% of sampled series"); axes[0].set_ylim(0, 100)
axes[0].set_title("Verdict Breakdown per Stratum")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)

# Right: CI plot
ps  = [r["p"] * 100 for r in ci_rows]
los = [r["p"] * 100 - r["lo"] * 100 for r in ci_rows]
his = [r["hi"] * 100 - r["p"] * 100 for r in ci_rows]
axes[1].barh(labels, ps, xerr=[los, his], capsize=5,
             color="steelblue", edgecolor="black", error_kw={"elinewidth": 1.5})
axes[1].axvline(50, color="red", linestyle="--", linewidth=1, label="50% threshold")
axes[1].set_xlabel("% Non-stationary or Conflicting"); axes[1].set_xlim(0, 100)
axes[1].set_title("95% Wilson CI — Non-stationarity Rate per Stratum")
axes[1].legend(); axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("ca_stationarity_ci.png", dpi=150)
plt.show()


## 8. ACF / PACF — Lag Structure

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Use total CA daily sales — aggregate is smoother and shows shared patterns
agg_series = df_CA.groupby("date", observed=True)["sales"].sum().sort_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
plot_acf(agg_series,  lags=56, ax=axes[0], title="ACF — Total CA Daily Sales (lags 0–56)")
plot_pacf(agg_series, lags=56, ax=axes[1], title="PACF — Total CA Daily Sales (lags 0–56)", method="ywm")

for ax in axes:
    # Mark the key weekly lags
    for lag in [7, 14, 21, 28]:
        ax.axvline(lag, color="red", linestyle="--", alpha=0.4, linewidth=0.8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ca_acf_pacf.png", dpi=150)
plt.show()
print("Red dashed lines mark lags 7, 14, 21, 28 — spikes confirm weekly seasonality")

## 9. STL Decomposition — Trend / Seasonality / Residual

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(agg_series, period=7, robust=True)
res = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
components = [
    (agg_series,      "Observed",   "steelblue"),
    (res.trend,       "Trend",      "darkorange"),
    (res.seasonal,    "Seasonality","green"),
    (res.resid,       "Residual",   "red"),
]
for ax, (data, label, color) in zip(axes, components):
    ax.plot(data, linewidth=0.8, color=color)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

axes[0].set_title("STL Decomposition — Total CA Daily Sales (period=7)")
plt.tight_layout()
plt.savefig("ca_stl_decomposition.png", dpi=150)
plt.show()

## 10. Day-of-Week Effect

In [ ]:
day_order = ["Saturday", "Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

dow = (
    df_CA.groupby(["weekday", "cat_id"], observed=True)["sales"]
    .mean()
    .reset_index()
)
dow["weekday"] = pd.Categorical(dow["weekday"], categories=day_order, ordered=True)
dow = dow.sort_values("weekday")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, (cat, grp) in zip(axes, dow.groupby("cat_id", observed=True)):
    ax.bar(grp["weekday"], grp["sales"], color="steelblue", edgecolor="black")
    ax.set_title(f"{cat}")
    ax.set_xlabel("Day of Week")
    ax.set_ylabel("Avg Units Sold")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Day-of-Week Effect by Category — California", fontsize=13)
plt.tight_layout()
plt.savefig("ca_dow_effect.png", dpi=150)
plt.show()

## 11. Zero-Sales / Intermittency Analysis

In [ ]:
zero_rate = (
    df_CA.groupby(["item_id", "store_id"], observed=True)
    .apply(lambda x: (x["sales"] == 0).mean(), include_groups=False)
    .reset_index(name="zero_rate")
)

print("Zero-sales rate distribution:")
print(zero_rate["zero_rate"].describe().round(3))

thresholds = [0.5, 0.8, 0.9]
for t in thresholds:
    pct = (zero_rate["zero_rate"] > t).mean() * 100
    print(f"  Series with >{int(t*100)}% zero days: {pct:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(zero_rate["zero_rate"], bins=50, color="steelblue", edgecolor="black")
axes[0].axvline(0.8, color="red", linestyle="--", label=">80% zeros (intermittent)")
axes[0].set_title("Distribution of Zero-Sales Rate per Series")
axes[0].set_xlabel("Fraction of days with zero sales")
axes[0].set_ylabel("Number of series")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Zero rate by category
zero_by_cat = zero_rate.merge(
    df_CA[["item_id", "store_id", "cat_id"]].drop_duplicates(),
    on=["item_id", "store_id"]
)
zero_by_cat.boxplot(column="zero_rate", by="cat_id", ax=axes[1])
axes[1].set_title("Zero-Sales Rate by Category")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Zero-sales rate")
plt.suptitle("")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ca_zero_sales.png", dpi=150)
plt.show()

## 12. Price Sensitivity

In [ ]:
df_priced = df_CA.dropna(subset=["sell_price"])

# Correlation between sell_price and sales per category
print("Pearson correlation (sell_price vs sales) by category:")
print(
    df_priced.groupby("cat_id", observed=True)
    .apply(lambda x: x["sell_price"].corr(x["sales"]), include_groups=False)
    .round(3)
)

# Price change detection — flag weeks where price shifts >10% vs prior week
price_changes = (
    df_CA[["item_id", "store_id", "wm_yr_wk", "sell_price"]]
    .drop_duplicates()
    .sort_values(["item_id", "store_id", "wm_yr_wk"])
)
price_changes["prev_price"] = price_changes.groupby(
    ["item_id", "store_id"], observed=True
)["sell_price"].shift(1)
price_changes["pct_change"] = (
    (price_changes["sell_price"] - price_changes["prev_price"]) / price_changes["prev_price"]
)
big_drops = price_changes[price_changes["pct_change"] < -0.10]
print(f"\nPrice drops >10% in a single week: {len(big_drops):,} occurrences across {big_drops['item_id'].nunique():,} items")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for cat, grp in df_priced.groupby("cat_id", observed=True):
    axes[0].scatter(grp["sell_price"], grp["sales"], alpha=0.01, s=1, label=cat)
axes[0].set_title("Sell Price vs Sales by Category")
axes[0].set_xlabel("Sell Price ($)")
axes[0].set_ylabel("Units Sold")
axes[0].legend(markerscale=8)
axes[0].grid(True, alpha=0.3)

price_changes["pct_change"].dropna().clip(-0.5, 0.5).hist(bins=60, ax=axes[1], color="darkorange", edgecolor="black")
axes[1].axvline(-0.10, color="red", linestyle="--", label=">10% drop threshold")
axes[1].set_title("Week-over-Week Price Change Distribution")
axes[1].set_xlabel("% Change in Sell Price")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ca_price_sensitivity.png", dpi=150)
plt.show()

## 13. Event Effects by Type

In [ ]:
baseline = df_CA[df_CA["event_type_1"] == "No_Event"]["sales"].mean()

event_effect = (
    df_CA[df_CA["event_type_1"] != "No_Event"]
    .groupby(["event_type_1", "cat_id"], observed=True)["sales"]
    .mean()
    .reset_index()
)
event_effect["lift_pct"] = ((event_effect["sales"] - baseline) / baseline * 100).round(1)

print(f"Baseline avg sales (no event): {baseline:.2f}")
print("\nSales lift % on event days by event type and category:")
print(event_effect.pivot(index="event_type_1", columns="cat_id", values="lift_pct").to_string())

event_types = event_effect["event_type_1"].unique()
fig, axes = plt.subplots(1, len(event_types), figsize=(14, 4), sharey=False)

for ax, etype in zip(axes, event_types):
    grp = event_effect[event_effect["event_type_1"] == etype]
    colors = ["green" if v >= 0 else "red" for v in grp["lift_pct"]]
    ax.bar(grp["cat_id"].astype(str), grp["lift_pct"], color=colors, edgecolor="black")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(etype)
    ax.set_xlabel("Category")
    ax.set_ylabel("Lift %")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Sales Lift % on Event Days vs Baseline (No Event) — CA", fontsize=12)
plt.tight_layout()
plt.savefig("ca_event_effects.png", dpi=150)
plt.show()

## 14. Store-Level Correlation

In [ ]:
store_daily_pivot = (
    df_CA.groupby(["date", "store_id"], observed=True)["sales"]
    .sum()
    .unstack("store_id")
)

corr = store_daily_pivot.corr()
print("Store-to-store correlation (daily sales):")
print(corr.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

im = axes[0].imshow(corr, vmin=0, vmax=1, cmap="YlOrRd")
axes[0].set_xticks(range(len(corr))); axes[0].set_xticklabels(corr.columns)
axes[0].set_yticks(range(len(corr))); axes[0].set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        axes[0].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=axes[0])
axes[0].set_title("Store Correlation Heatmap")

store_daily_pivot.plot(ax=axes[1], linewidth=0.8, alpha=0.8)
axes[1].set_title("Daily Sales per Store — CA")
axes[1].set_xlabel("Date"); axes[1].set_ylabel("Units Sold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ca_store_correlation.png", dpi=150)
plt.show()

## 15. Feature Engineering — Lag Features

In [ ]:
import calendar

# ── Step 1: Sort ──────────────────────────────────────────────────────────────
print("Step 1: Sorting by item_id, store_id, date...")
df_CA = df_CA.sort_values(["item_id", "store_id", "date"]).reset_index(drop=True)
print(f"  Shape: {df_CA.shape}")

# ── Step 2a: lag_7 and lag_28 ─────────────────────────────────────────────────
print("Step 2a: Computing lag_7 and lag_28...")
grp = df_CA.groupby(["item_id", "store_id"], observed=True)["sales"]
df_CA["lag_7"]  = grp.shift(7).astype("float32")
df_CA["lag_28"] = grp.shift(28).astype("float32")

# ── Step 2b: lag_364 — leap-year aware ───────────────────────────────────────
# Look back 366 days if previous calendar year was a leap year, else 365.
# This lands on the same calendar date last year regardless of leap years,
# while still preserving day-of-week alignment in non-leap years.
print("Step 2b: Computing lag_364 (leap-year aware)...")

prev_year = df_CA["date"].dt.year - 1
lookback_days = prev_year.map(lambda y: 366 if calendar.isleap(y) else 365)
df_CA["_lag_date"] = df_CA["date"] - pd.to_timedelta(lookback_days, unit="D")

lookup = (
    df_CA[["item_id", "store_id", "date", "sales"]]
    .rename(columns={"date": "_lag_date", "sales": "lag_364"})
)
df_CA = df_CA.merge(lookup, on=["item_id", "store_id", "_lag_date"], how="left")
df_CA["lag_364"] = df_CA["lag_364"].astype("float32")
df_CA = df_CA.drop(columns=["_lag_date"])

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Memory: {df_CA.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("Null counts (expected — first N days have no lag):")
print(df_CA[["lag_7", "lag_28", "lag_364"]].isnull().sum())
print("Sample (one item x store):")
sample_check = (
    df_CA[df_CA["item_id"] == df_CA["item_id"].iloc[0]]
    [["date", "sales", "lag_7", "lag_28", "lag_364"]]
    .head(10)
)
print(sample_check.to_string(index=False))


In [ ]:
sample_check = (
    df_CA[df_CA["item_id"] == df_CA["item_id"].iloc[0]]
    [["date", "sales", "lag_7", "lag_28", "lag_364"]]
    .head(366)
)